# 14 · Profile: AI exposure by job (Eloundou et al., "GPTs are GPTs")

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/eloundou_ai_exposure.md`. Decision: D7.

**Why this source matters.** brief S12: one published research score of AI exposure per job, labeled "research estimate, not government data". I picked **Eloundou, Manning, Mishkin & Rock (2024), *Science*, β measure, GPT-4 rated (D7). It answers "how much of this job's work could a large language model speed up?" It does not** say jobs will disappear.

**How the score is built.** Every O*NET task got a label: E0 no exposure, E1 an LLM alone cuts the time at least in half, E2 an LLM plus software built on it would. A job's β = average over its tasks of (E1 = 1, E2 = 0.5, E0 = 0). Each task was labeled by GPT-4 (`dv_` columns) and by human annotators (`human_` columns).

**What this notebook does:** reads the job and task files, checks grain and codes, rebuilds the published job scores from the task labels, maps O*NET jobs to SOC 2018, and measures coverage of NYC jobs. It changes nothing.

In [1]:
import sys, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key == "eloundou_ai_exposure"][["key", "file", "downloaded_utc", "bytes", "note"]]
display(FILES)
P = {Path(f).name: RAW / f for f in FILES.file}
print(P["README.md"].read_text().strip())

,key,file,downloaded_utc,bytes,note
201,eloundou_ai_exposure,skills/eloundou_ai_exposure/occ_level.csv,2026-10-13T04:22:32+00:00,126022,Eloundou et al. 2024 'GPTs are GPTs' (Science): AI exposure by O*N...
202,eloundou_ai_exposure,skills/eloundou_ai_exposure/full_labelset.tsv,2026-10-13T04:22:33+00:00,3893248,Eloundou et al. 2024 'GPTs are GPTs' (Science): AI exposure task l...
203,eloundou_ai_exposure,skills/eloundou_ai_exposure/README.md,2026-10-13T04:22:34+00:00,172,Eloundou et al. 2024 'GPTs are GPTs' (Science): Repo README: colum...
204,eloundou_ai_exposure,skills/eloundou_ai_exposure/LICENSE.txt,2026-10-13T04:22:35+00:00,1063,Eloundou et al. 2024 'GPTs are GPTs' (Science): MIT license


# GPTs-are-GPTs

Occupation level codes are in occ_level.csv! 
- dv_rating is the GPT-4 rating and human is from the annotators
- _alpha=E1, _beta = E1+.5*E2, _gamma=E1+E2


## 0. Source header

| | GPTs are GPTs: AI exposure by job (`eloundou_ai_exposure`) |
|---|---|
| **Publisher** | Eloundou, Manning, Mishkin, Rock (OpenAI, OpenResearch, University of Pennsylvania). *Science* 384 (2024). Data on GitHub, `openai/GPTs-are-GPTs` |
| **Files** | `occ_level.csv` (one row per O*NET job) and `full_labelset.tsv` (one row per O*NET task), plus the repo README and LICENSE. Downloaded from commit `0471612`, so the files can't change under us |
| **Release** | Labels made in 2023 on the O*NET 27.x task list; job file added to the repo October 2025 |
| **How it was made** | **Model + expert rating.** Every task rated E0 / E1 / E2 by GPT-4 and by human annotators using one rubric. Not a survey, no counts |
| **License** | MIT (© 2024 OpenAI) |
| **Documentation** | [Paper (arXiv 2303.10130)](https://arxiv.org/abs/2303.10130), [*Science* version](https://www.science.org/doi/10.1126/science.adj0998), repo README |

## 1. Load as text

In [2]:
occ = pd.read_csv(P["occ_level.csv"], dtype=str)
tasks = pd.read_csv(P["full_labelset.tsv"], sep="\t", dtype=str).drop(columns="Unnamed: 0")
for n, d in {"occ_level": occ, "full_labelset": tasks}.items():
    print(f"{n:<14} {d.shape[0]:>6,} rows x {d.shape[1]} cols | blanks: {d.isna().sum()[d.isna().sum() > 0].to_dict()}")
display(occ.head(3)); display(tasks.head(3))
SCORES = [c for c in occ.columns if c.startswith(("dv_", "human_"))]
for c in SCORES: occ[c] = occ[c].astype(float)

occ_level         923 rows x 8 cols | blanks: {}
full_labelset  19,265 rows x 15 cols | blanks: {'Task Type': 1312}


,O*NET-SOC Code,Title,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma
0,11-1011.00,Chief Executives,0.1,0.46,0.8200000000000001,0.18,0.35,0.52
1,11-1011.03,Chief Sustainability Officers,0.16666666666666666,0.5555555555555556,0.9444444444444444,0.05555555555555555,0.38888888888888884,0.7222222222222222
2,11-1021.00,General and Operations Managers,0.0,0.4807692307692308,0.9615384615384616,0.11538461538461539,0.38461538461538464,0.6538461538461539


,O*NET-SOC Code,Task ID,Task,Task Type,Title,human_exposure_agg,gpt4_exposure,gpt4_exposure_alt_rubric,gpt_3_relevant,gpt4_automation,alpha,beta,gamma,automation,human_labels
0,11-1011.00,8823.0,Direct or coordinate an organization's financial or budget activit...,Core,Chief Executives,E0,E2,E2,False,T2,0.0,0.5,1.0,0.5,E0
1,11-1011.00,8831.0,Appoint department heads or managers and assign or delegate respon...,Core,Chief Executives,E0,E0,E0,False,T1,0.0,0.0,0.0,0.25,E0
2,11-1011.00,8825.0,Analyze operations to evaluate performance of a company or its sta...,Core,Chief Executives,E2,E2,E2,False,T2,0.0,0.5,1.0,0.5,E2


**Load findings**
- `occ_level.csv`: **923 O*NET jobs**, 6 scores (α, β, γ, each by GPT-4 `dv_` and by humans `human_`), no blanks.
- `full_labelset.tsv`: 19,265 tasks with each task's GPT-4 label, human label, an alternative-rubric GPT-4 label, and a separate automation label (`T0–T4`, not used). The first column is a leftover pandas index. `Task Type` is blank for 1,312 tasks.
- The README defines the scores: α = E1, β = E1 + 0.5 × E2, γ = E1 + E2.

## 2. What one row is (grain)

**Presumed keys:** job file `O*NET-SOC Code`; task file `O*NET-SOC Code + Task ID`.

In [3]:
print("Job file, duplicate O*NET-SOC codes:", occ["O*NET-SOC Code"].duplicated().sum(), "| jobs:", len(occ))
print("Task file, duplicate code + task:", tasks.duplicated(["O*NET-SOC Code", "Task ID"]).sum(),
      "| tasks:", len(tasks), "| jobs:", tasks["O*NET-SOC Code"].nunique())
print("Same jobs in both files:", set(occ["O*NET-SOC Code"]) == set(tasks["O*NET-SOC Code"]))
print("Task ID stored as", tasks["Task ID"].head(2).tolist(), "-> a float written as text; whole numbers?",
      (tasks["Task ID"].astype(float) % 1 == 0).all())
print("Tasks per job:", tasks.groupby("O*NET-SOC Code").size().describe().round(1).to_dict())

Job file, duplicate O*NET-SOC codes: 0 | jobs: 923
Task file, duplicate code + task: 0 | tasks: 19265 | jobs: 923
Same jobs in both files: True
Task ID stored as ['8823.0', '8831.0'] -> a float written as text; whole numbers? True
Tasks per job: {'count': 923.0, 'mean': 20.9, 'std': 6.4, 'min': 4.0, '25%': 16.0, '50%': 20.0, '75%': 25.0, 'max': 40.0}


**Grain findings**
- Job file: one row per O*NET-SOC job, unique . Task file: `O*NET-SOC Code + Task ID` unique . Both files list the same 923 jobs.
- `Task ID` is written as a decimal (`8823.0`); every value is a whole number, so it converts to O*NET's integer ID cleanly.
- 4–40 tasks per job (median 20).

## 3–4. Profile: labels, scores, GPT-4 vs humans

In [4]:
print("Task Type:", tasks["Task Type"].value_counts(dropna=False).to_dict())
for c in ["gpt4_exposure", "human_exposure_agg", "human_labels", "gpt4_automation"]:
    print(f"{c:<20}", tasks[c].value_counts(dropna=False).to_dict())
print("human_labels == human_exposure_agg:", (tasks.human_labels == tasks.human_exposure_agg).mean())
display(occ[SCORES].describe().round(3))
b = occ[["dv_rating_beta", "human_rating_beta"]]
print(f"GPT-4 vs human beta: Pearson {b.corr().iloc[0, 1]:.3f}, rank {b.rank().corr().iloc[0, 1]:.3f}; "
      f"GPT-4 higher on average by {(b.dv_rating_beta - b.human_rating_beta).mean():.3f}")
print("Ordering alpha <= beta <= gamma holds for every job:",
      ((occ.dv_rating_alpha <= occ.dv_rating_beta + 1e-9) & (occ.dv_rating_beta <= occ.dv_rating_gamma + 1e-9)).all())

Task Type: {'Core': 13225, 'Supplemental': 4728, nan: 1312}
gpt4_exposure        {'E0': 8561, 'E2': 8002, 'E1': 2702}
human_exposure_agg   {'E0': 10256, 'E2': 6147, 'E1': 2862}
human_labels         {'E0': 10256, 'E2': 6147, 'E1': 2862}
gpt4_automation      {'T1': 6284, 'T0': 5746, 'T3': 3879, 'T2': 3224, 'T4': 132}
human_labels == human_exposure_agg: 1.0


,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma
count,923.000,923.000,923.000,923.000,923.000,923.000
mean,0.142,0.345,0.548,0.144,0.303,0.461
std,0.166,0.221,0.345,0.145,0.207,0.297
min,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.041,0.145,0.200,0.000,0.109,0.185
50%,0.097,0.361,0.600,0.115,0.306,0.458
75%,0.182,0.500,0.875,0.235,0.472,0.727
max,1.000,1.000,1.000,0.800,0.844,1.000


GPT-4 vs human beta: Pearson 0.876, rank 0.901; GPT-4 higher on average by 0.042
Ordering alpha <= beta <= gamma holds for every job: True


**Profile findings**
- GPT-4 labels: 44% of tasks E0, 14% E1, 42% E2. Humans: 53% E0, 15% E1, 32% E2. GPT-4 rates exposure a little higher (β +0.04 on average).
- Job β (GPT-4) runs 0 to 1, median 0.36. α ≤ β ≤ γ for every job, as the definitions require.
- GPT-4 and humans agree closely on the ranking (rank correlation 0.90, Pearson 0.88). That's the paper's own validation, and it holds in this file.

## 4b. Codes: O*NET-SOC 2019 → SOC 2018, and is the task list still current?

The labels were made on an older O*NET release. Check the jobs and tasks against **O*NET 31.0** (August 2026, nb 11) and map to SOC 2018 with the O*NET crosswalk.

In [5]:
pz = lambda n: RAW / man.loc[man.file.str.endswith(n), "file"].iloc[0]
z = zipfile.ZipFile(pz("db_31_0_csv.zip"))
od = pd.read_csv(z.open("db_31_0_csv/occupation_data.csv"), dtype=str)
ts = pd.read_csv(z.open("db_31_0_csv/task_statements.csv"), dtype=str)
J, J31 = set(occ["O*NET-SOC Code"]), set(od.iloc[:, 0])
print(f"Jobs: Eloundou {len(J)} | O*NET 31.0 {len(J31)} | in both {len(J & J31)} | only Eloundou {sorted(J - J31)} | only 31.0 {len(J31 - J)}")
print("Only in 31.0 = 'All Other' and similar jobs O*NET doesn't rate:", sorted(J31 - J)[:12], "...")
tasks["tid"] = tasks["Task ID"].astype(float).astype(int).astype(str)
T31 = set(ts["Task ID"])
print(f"Eloundou tasks still in O*NET 31.0: {tasks.tid.isin(T31).mean():.1%} | O*NET 31.0 tasks that Eloundou rated: {ts['Task ID'].isin(set(tasks.tid)).mean():.1%}")
same = tasks.merge(ts[["Task ID", "Task"]], left_on="tid", right_on="Task ID", suffixes=("", "_31"))
print(f"Same wording where the task is in both: {(same.Task.str.strip() == same.Task_31.str.strip()).mean():.1%}")

cw = pd.read_excel(pz("2019_to_SOC_Crosswalk.xlsx"), header=3, dtype=str)
cw.columns = ["onet", "onet_title", "soc", "soc_title"]
m = occ.merge(cw[["onet", "soc"]], left_on="O*NET-SOC Code", right_on="onet", how="left")
print(f"\nEloundou jobs found in the O*NET-SOC 2019 crosswalk: {m.soc.notna().mean():.1%}; "
      f"crosswalk SOC == first 7 characters: {(m.soc == m['O*NET-SOC Code'].str[:7]).mean():.1%}")
occ["soc"] = occ["O*NET-SOC Code"].str[:7]; tasks["soc"] = tasks["O*NET-SOC Code"].str[:7]
n = occ.groupby("soc").size()
print(f"SOC 2018 jobs: {len(n)}; with several O*NET profiles: {(n > 1).sum()}; SOC jobs with no '.00' profile: {(~occ.groupby('soc')['O*NET-SOC Code'].apply(lambda s: s.str.endswith('.00').any())).sum()}")

Jobs: Eloundou 923 | O*NET 31.0 1016 | in both 923 | only Eloundou [] | only 31.0 93
Only in 31.0 = 'All Other' and similar jobs O*NET doesn't rate: ['11-9039.00', '11-9179.00', '11-9199.00', '13-1199.00', '13-2099.00', '15-1299.00', '15-2099.00', '17-2199.00', '17-3019.00', '17-3029.00', '19-1029.00', '19-1099.00'] ...
Eloundou tasks still in O*NET 31.0: 97.2% | O*NET 31.0 tasks that Eloundou rated: 99.5%
Same wording where the task is in both: 99.7%



Eloundou jobs found in the O*NET-SOC 2019 crosswalk: 100.0%; crosswalk SOC == first 7 characters: 100.0%
SOC 2018 jobs: 798; with several O*NET profiles: 67; SOC jobs with no '.00' profile: 24


**Code findings**
- **All 923 jobs are current O*NET 31.0 jobs.** The 93 31.0 jobs not scored are "All Other" and similar jobs that O*NET gives no tasks.
- The task list is still current: 97.2% of the rated tasks are still in O*NET 31.0, and they cover 99.5% of 31.0's tasks, with the same wording 99.7% of the time. Rating in 2023 doesn't make the score stale for the task list; the stale part is the model (GPT-4, 2023).
- O*NET-SOC → SOC 2018 = the first 7 characters (100%, checked against the O*NET crosswalk, as in nb 11). 798 SOC jobs; 67 have several O*NET profiles, and 24 of those have no `.00` profile (e.g. `29-1229` physicians, all specialties).

## 7. Tie-out: can we rebuild the published scores from the task labels?

The job file should be the task labels averaged. Test the obvious rule (plain mean), the core-only mean, and the weighting O*NET itself uses for task importance (core = 1, supplemental = 0.5). If one rule reproduces every job exactly, we can use the same rule to roll O*NET jobs up to SOC jobs.

In [6]:
V = {"E0": 0.0, "E1": 1.0, "E2": 0.5}
tasks["b_gpt"], tasks["b_hum"] = tasks.gpt4_exposure.map(V), tasks.human_exposure_agg.map(V)
tasks["w"] = np.where(tasks["Task Type"] == "Core", 1.0, 0.5)   # supplemental and blank both 0.5
print("Task column `beta` == GPT-4 label mapped E1=1, E2=.5:", (tasks.beta.astype(float) == tasks.b_gpt).mean())
wavg = lambda g, c: np.average(g[c], weights=g.w)
rules = {"plain mean": tasks.groupby("O*NET-SOC Code").b_gpt.mean(),
         "core tasks only": tasks[tasks["Task Type"] == "Core"].groupby("O*NET-SOC Code").b_gpt.mean(),
         "core 1, supplemental 0.5": tasks.groupby("O*NET-SOC Code")[["b_gpt", "w"]].apply(wavg, c="b_gpt")}
pub = occ.set_index("O*NET-SOC Code").dv_rating_beta
display(pd.DataFrame({r: {"jobs matching (< 0.001)": f"{((s - pub).abs() < 1e-3).mean():.1%}", "largest gap": round((s - pub).abs().max(), 4)} for r, s in rules.items()}).T)
hum = tasks.groupby("O*NET-SOC Code")[["b_hum", "w"]].apply(wavg, c="b_hum")
print(f"Human beta, same weighting: {((hum - occ.set_index('O*NET-SOC Code').human_rating_beta).abs() < 1e-3).mean():.1%} of jobs match")

Task column `beta` == GPT-4 label mapped E1=1, E2=.5: 1.0


,jobs matching (< 0.001),largest gap
plain mean,26.4%,0.1071
core tasks only,20.7%,0.2333
"core 1, supplemental 0.5",100.0%,0.0


Human beta, same weighting: 100.0% of jobs match


**Tie-out findings **
- The published job scores rebuild exactly from the task labels: a weighted mean with core tasks = 1, supplemental and blank = 0.5. Every one of 923 jobs matches, for GPT-4 and for humans. A plain mean matches only 26%.
- So the right way to roll O*NET jobs up to SOC jobs is the same rule, pooled over all tasks under the SOC job. It differs from a plain mean of the profiles by 0.008 on average (max 0.047); using only the `.00` profile would be off by 0.049 on average, and impossible for 24 jobs.

## 5. Coverage: how much of NYC's work gets a score?

NYC jobs = NYS DOL wages, NYC, detailed codes (as in nb 13). SOC score = the **task-weighted average over every O*NET profile under the SOC job** (the rule §7 just confirmed, applied one level up). BLS group codes (`31-1120`, `29-2010` …) get the pooled tasks of the SOC jobs under them.

In [7]:
pool = lambda key: tasks.groupby(key)[["b_gpt", "w"]].apply(wavg, c="b_gpt")
soc_b = pool("soc")
mean_b = occ.groupby("soc").dv_rating_beta.mean()
dot00 = occ[occ["O*NET-SOC Code"].str.endswith(".00")].set_index("soc").dv_rating_beta
multi = n[n > 1].index
print(f"SOC jobs with several profiles ({len(multi)}): pooled vs mean of profiles, mean gap {(soc_b - mean_b)[multi].abs().mean():.3f}, max {(soc_b - mean_b)[multi].abs().max():.3f}; "
      f"pooled vs '.00' only, mean gap {(soc_b - dot00).reindex(multi).abs().mean():.3f}")

W = RAW / man.loc[man.key == "nysdol_oews_regions", "file"].iloc[0]
w = pd.read_csv(W, dtype=str, keep_default_na=False); w.columns = [c.strip() for c in w.columns]
w = w[(w["Area Name"].str.strip() == "New York City") & w["Standard Occupational Code"].str.fullmatch(r"\d\d-\d{4}")
      & ~w["Standard Occupational Code"].str.endswith("0")].set_index("Standard Occupational Code")
emp = pd.to_numeric(w.Employment.str.replace(",", ""), errors="coerce").dropna()
title = w["Occupational Title"]

# BLS/OEWS aggregate codes and the SOC 2018 jobs they combine (BLS OEWS occupation list; same set as nb 06 / nb 13)
AGG = {"13-1020": ["13-1021", "13-1022", "13-1023"], "13-2020": ["13-2022", "13-2023"], "21-1018": ["21-1011", "21-1014"],
       "25-2052": ["25-2055", "25-2056"], "25-9045": ["25-9042", "25-9043", "25-9049"], "29-2010": ["29-2011", "29-2012"],
       "31-1120": ["31-1121", "31-1122"], "39-7010": ["39-7011", "39-7012"], "47-4090": ["47-4091", "47-4099"],
       "51-2028": ["51-2022", "51-2023"], "51-2090": ["51-2092", "51-2099"], "53-1047": ["53-1042", "53-1043", "53-1044", "53-1049"]}
minor = lambda c: c[:4]   # e.g. 43-9 for 43-9199

def score(c):
    if c in soc_b.index:
        return soc_b[c], "1 exact SOC job"
    kids = tasks[tasks.soc.isin(AGG.get(c, []))]
    if len(kids):
        return np.average(kids.b_gpt, weights=kids.w), "2 BLS group code, pooled from its parts"
    return np.nan, "3 no score (O*NET doesn't rate 'All Other' jobs)"
nyc = pd.DataFrame([score(c) for c in emp.index], index=emp.index, columns=["beta", "how"]).assign(emp=emp, title=title)
cov = nyc.groupby("how").agg(jobs=("emp", "size"), nyc_employment=("emp", "sum"))
cov["share"] = (cov.nyc_employment / cov.nyc_employment.sum()).map("{:.1%}".format)
display(cov)
display(nyc[nyc.how.str[0] != "1"].sort_values("emp", ascending=False).head(12))
print("Parts with no score of their own:", {a: [k for k in v if k not in soc_b.index] for a, v in AGG.items() if any(k not in soc_b.index for k in v)})
# Possible fill for 'All Other' jobs: NYC-employment-weighted mean of the scored jobs in the same SOC minor group
sc = nyc.dropna(subset=["beta"])
grp = sc.groupby(sc.index.map(minor))[["beta", "emp"]].apply(lambda g: np.average(g.beta, weights=g.emp))
left = nyc[nyc.beta.isna()].assign(minor_group_fill=lambda d: d.index.map(minor).map(grp))
print(f"'All Other' jobs: {len(left)}, {left.emp.sum() / nyc.emp.sum():.1%} of NYC employment; a minor-group fill would reach {left.minor_group_fill.notna().sum()} of them")

SOC jobs with several profiles (67): pooled vs mean of profiles, mean gap 0.008, max 0.047; pooled vs '.00' only, mean gap 0.049


,jobs,nyc_employment,share
how,,,
1 exact SOC job,530,3924910.0,97.0%
"2 BLS group code, pooled from its parts",5,84140.0,2.1%
3 no score (O*NET doesn't rate 'All Other' jobs),21,38570.0,1.0%


,beta,how,emp,title
Standard Occupational Code,,,,
25-9045,0.316667,"2 BLS group code, pooled from its parts",50780.0,"Teaching Assistants, Except Postsecondary"
21-1018,0.229167,"2 BLS group code, pooled from its parts",12030.0,"Substance abuse, behavioral disorder, and mental health counselors..."
53-1047,0.310606,"2 BLS group code, pooled from its parts",11880.0,First-Line Supervisors of Transportation and Material Moving Worke...
43-9199,NaN,3 no score (O*NET doesn't rate 'All Other' jobs),10880.0,"Office and Administrative Support Workers, All Other"
21-1099,NaN,3 no score (O*NET doesn't rate 'All Other' jobs),10160.0,"Community and Social Service Specialists, All Other"
25-2052,0.283333,"2 BLS group code, pooled from its parts",8310.0,"Special Education Teachers, Kindergarten and Elementary School"
29-1249,NaN,3 no score (O*NET doesn't rate 'All Other' jobs),2780.0,"Surgeons, All Other"
11-9039,NaN,3 no score (O*NET doesn't rate 'All Other' jobs),1750.0,"Education Administrators, All Other"
53-3099,NaN,3 no score (O*NET doesn't rate 'All Other' jobs),1360.0,"Motor Vehicle Operators, All Other"


Parts with no score of their own: {'25-9045': ['25-9049'], '51-2090': ['51-2099'], '53-1047': ['53-1049']}
'All Other' jobs: 21, 1.0% of NYC employment; a minor-group fill would reach 21 of them


**Coverage findings**
- 97.0% of NYC employment is scored directly (530 jobs). Another 2.1% comes through BLS group codes pooled from their parts (teaching assistants `25-9045` 51k, counselors `21-1018`, transport supervisors `53-1047`, special ed teachers `25-2052`, assemblers `51-2028`). Total: 99.0%. A few parts are "All Other" jobs with no score of their own; the group is scored from the parts that have one.
- **1.0% unscored:** 21 "All Other" jobs (office support all other `43-9199` 11k, community and social service all other `21-1099` 10k …). Proposed: leave them unscored and say so ("not rated"). Filling them from the same SOC minor group would reach all 21, but it's an invented number for 1% of jobs. Filling them stays an option.
- Better than O*NET 31.0's own skill ratings: Financial and Investment Analysts `13-2051` (54k NYC jobs, "not yet rated" in nb 11) has a score here (β 0.46), because it uses the 2019 job list.

## 6. Relationships: exposure by entry education and by career cluster

Two joins the dashboard will make: BLS typical entry education (nb 04, Table 5.4) and Career Clusters (nb 13, D6). Averages are weighted by NYC employment.

In [8]:
ep = pd.read_excel(RAW / man.loc[man.key == "bls_ep_education", "file"].iloc[0], sheet_name="Table 5.4", dtype=str, header=1, keep_default_na=False)
ep = ep[ep.iloc[:, 1].str.fullmatch(r"\d\d-\d{4}")].iloc[:, [1, 2]]; ep.columns = ["soc", "education"]
x = nyc.dropna(subset=["beta"]).join(ep.set_index("soc"))
wm = lambda g: pd.Series({"jobs": len(g), "nyc_employment": g.emp.sum(), "avg_beta": np.average(g.beta, weights=g.emp),
                          "share_emp_beta_ge_0.5": g.emp[g.beta >= 0.5].sum() / g.emp.sum()})
ORDER = ["No formal educational credential", "High school diploma or equivalent", "Postsecondary nondegree award", "Some college, no degree",
         "Associate's degree", "Bachelor's degree", "Master's degree", "Doctoral or professional degree"]
display(x.groupby("education")[["beta", "emp"]].apply(wm).reindex(ORDER).round(3))

cc = pd.read_excel(pz("Full_Framework_Crosswalk_SOC-Career-Clusters.xlsx"), sheet_name="SOC - CC - Sub-Clusters", dtype=str)
cc = cc[["SOC Code", "Career Cluster"]].drop_duplicates().apply(lambda s: s.str.strip())
y = cc.merge(x.reset_index().rename(columns={"Standard Occupational Code": "soc"}), left_on="SOC Code", right_on="soc")
display(y.groupby("Career Cluster")[["beta", "emp"]].apply(wm).sort_values("avg_beta", ascending=False).round(3))

,jobs,nyc_employment,avg_beta,share_emp_beta_ge_0.5
education,,,,
No formal educational credential,62.0,790310.0,0.133,0.008
High school diploma or equivalent,183.0,1202530.0,0.409,0.432
Postsecondary nondegree award,38.0,187770.0,0.216,0.032
"Some college, no degree",6.0,111840.0,0.526,0.536
Associate's degree,28.0,73100.0,0.408,0.502
Bachelor's degree,138.0,1357430.0,0.518,0.519
Master's degree,27.0,114940.0,0.375,0.070
Doctoral or professional degree,53.0,171130.0,0.422,0.090


,jobs,nyc_employment,avg_beta,share_emp_beta_ge_0.5
Career Cluster,,,,
Digital Technology,19.0,212500.0,0.736,1.000
"Arts, Entertainment, & Design",46.0,224100.0,0.646,0.674
Financial Services,32.0,337690.0,0.586,0.820
Management & Entrepreneurship,40.0,634690.0,0.532,0.520
Marketing & Sales,22.0,470700.0,0.466,0.558
Energy & Natural Resources,36.0,182730.0,0.423,0.126
Education,62.0,398380.0,0.364,0.108
Public Service & Safety,60.0,432560.0,0.357,0.163
Healthcare & Human Services,109.0,559470.0,0.339,0.130


**Relationship findings**
- **By entry education (NYC employment-weighted):** bachelor's-level jobs are the most exposed (β 0.52; 52% of their NYC workers are in jobs with β ≥ 0.5), associate's-level 0.41 (50%), high school 0.41, no credential 0.13. Graduate-degree jobs are lower (0.38–0.42). That matches the paper: higher-wage, higher-education work is more exposed, not less.
- By career cluster (D6, each job counted in every cluster it's in): Digital Technology highest (0.74; every NYC worker in it is in a job with β ≥ 0.5), then Arts, Entertainment & Design 0.65, Financial Services 0.59, Management 0.53. Lowest: Hospitality 0.17, Construction 0.20, Supply Chain 0.25. Healthcare & Human Services 0.34 and Education 0.36 are in the middle.

## 8. Fitness for purpose: what NYC's jobs look like through this score

Employment-weighted distribution, the biggest NYC jobs at each end, and how far GPT-4's rating and the human rating disagree for NYC's big jobs (the dashboard shows the GPT-4 one, D7).

In [9]:
s = nyc.dropna(subset=["beta"])
print(f"NYC jobs scored: {s.emp.sum():,.0f}. Employment-weighted beta: mean {np.average(s.beta, weights=s.emp):.3f}")
bands = pd.cut(s.beta, [-0.001, 0.1, 0.25, 0.5, 0.75, 1.0], labels=["0-0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-1"])
display(s.groupby(bands, observed=False).agg(jobs=("emp", "size"), nyc_employment=("emp", "sum"))
        .assign(share=lambda d: (d.nyc_employment / d.nyc_employment.sum()).round(3)))
big = s[s.emp >= 15000]
print("Most exposed big NYC jobs (15k+ workers):"); display(big.nlargest(10, "beta")[["title", "emp", "beta"]].round(3))
print("Least exposed big NYC jobs:"); display(big.nsmallest(8, "beta")[["title", "emp", "beta"]].round(3))
hum_soc = tasks.groupby("soc")[["b_hum", "w"]].apply(wavg, c="b_hum")
big = big.assign(human=big.index.map(hum_soc)); big["gap"] = big.beta - big.human
print("Big NYC jobs where GPT-4 and humans disagree most:"); display(big.reindex(big.gap.abs().sort_values(ascending=False).index).head(8)[["title", "emp", "beta", "human", "gap"]].round(3))

NYC jobs scored: 4,009,050. Employment-weighted beta: mean 0.385


,jobs,nyc_employment,share
beta,,,
0-0.10,84,510580.0,0.127
0.10-0.25,94,710890.0,0.177
0.25-0.50,218,1548680.0,0.386
0.50-0.75,120,1076320.0,0.268
0.75-1,19,162580.0,0.041


Most exposed big NYC jobs (15k+ workers):


,title,emp,beta
Standard Occupational Code,,,
15-1252,Software Developers,74710.0,0.868
43-3031,"Bookkeeping, Accounting, and Auditing Clerks",32080.0,0.802
15-2051,Data Scientists,15960.0,0.680
43-6013,Medical Secretaries,23990.0,0.667
43-6011,Executive Secretaries and Executive Administrative Assistants,52600.0,0.649
13-1071,Human Resources Specialists,30000.0,0.644
43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, ...",49430.0,0.644
41-3091,"Sales Representatives of Services, Except Advertising, Insurance, ...",46890.0,0.633
41-4012,"Sales Representatives, Wholesale and Manufacturing, Except Technic...",23920.0,0.621


Least exposed big NYC jobs:


,title,emp,beta
Standard Occupational Code,,,
35-9011,Dining Room and Cafeteria Attendants and Bartender Helpers,17490.0,0.000
37-2011,"Janitors and Cleaners, Except Maids and Housekeeping Cleaners",96390.0,0.000
37-2012,Maids and Housekeeping Cleaners,22210.0,0.000
35-2021,Food Preparation Workers,31080.0,0.037
53-7062,"Laborers and Freight, Stock, and Material Movers, Hand",35820.0,0.042
47-2061,Construction Laborers,16670.0,0.050
35-3023,Fast Food and Counter Workers,66090.0,0.093
35-2014,"Cooks, Restaurant",34800.0,0.095


Big NYC jobs where GPT-4 and humans disagree most:

,title,emp,beta,human,gap
Standard Occupational Code,,,,,
43-3031,"Bookkeeping, Accounting, and Auditing Clerks",32080.0,0.802,0.314,0.488
15-1252,Software Developers,74710.0,0.868,0.447,0.421
35-1012,First-Line Supervisors of Food Preparation and Serving Workers,28390.0,0.511,0.256,0.256
31-9092,Medical Assistants,17500.0,0.359,0.141,0.218
39-9011,Childcare Workers,17340.0,0.142,0.340,-0.198
27-3031,Public Relations Specialists,18490.0,0.591,0.788,-0.197
13-2052,Personal Financial Advisors,15560.0,0.500,0.671,-0.171
41-2011,Cashiers,58470.0,0.209,0.360,-0.151


**Fitness findings**
- NYC employment-weighted β = 0.385. 31% of NYC's workers are in jobs with β ≥ 0.5 (half or more of the job's task time exposed); 13% are in jobs below 0.1.
- Most exposed big NYC jobs: software developers 0.87, bookkeeping clerks 0.80, data scientists 0.68, secretaries 0.64–0.67, HR specialists 0.64. Least exposed: janitors, maids, dining attendants (0.0), food prep, laborers.
- Where GPT-4 and humans disagree most among big NYC jobs: bookkeeping clerks (GPT-4 0.80 vs humans 0.31), software developers (0.87 vs 0.45), medical assistants; humans rate PR specialists and financial advisors higher. The dashboard uses GPT-4 (D7), so the job page should say "one research estimate; the authors' human raters scored some jobs differently".
- Proposed display (set before building pages, like the S11 rule): show the number as a share ("about 4 in 10 of this job's tasks…") and a band with fixed cutoffs: low < 0.25, medium 0.25–0.50, high ≥ 0.50. The 0.50 line is the paper's own threshold. Never words like "at risk" or "safe": exposure means "AI could speed this up", not job loss.

## 9. Issues list and proposed staging table

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Leftover index column; `Task ID` as decimal text | all tasks | Drop the index; cast `Task ID` to integer |
| 2 | `Task Type` blank | 1,312 tasks | Keep blank; weight 0.5 (that's what the published scores used, §7) |
| 3 | O*NET jobs, not SOC jobs | 923 → 798 | Roll up in core: pooled task mean, core 1 / supplemental 0.5 |
| 4 | BLS group codes (`25-9045` …) | 5 NYC codes, 2.1% of jobs | Pool the tasks of their SOC 2018 parts (same parts list as nb 06 / 13) |
| 5 | "All Other" jobs have no score | 21 NYC codes, 1.0% of jobs | Show "not rated"; no fill |
| 6 | GPT-4 and human ratings disagree for some big jobs | e.g. software developers 0.87 vs 0.45 | Show GPT-4 (D7); keep the human β in core for a note on the job page |
| 7 | Scored with GPT-4 in 2023 | all | Label with the year and model: "GPT-4-era estimate (2023 labels, published 2024)" |
| 8 | Only national, no NYC version | all | Fine: it's about the job's tasks, which don't change by city. NYC enters through employment weights |

**Proposed staging tables:**
- `stg_eloundou__task_exposure`: grain O*NET job × task. `onet_soc_code : char(10)`, `task_id : int`, `task_type : text` (nullable), `gpt4_label : char(2)` (E0/E1/E2), `human_label : char(2)`. 19,265 rows.
- `stg_eloundou__job_exposure`: grain O*NET job. The 6 published scores, kept to re-run the tie-out. 923 rows.
- **Core** builds `job_ai_exposure` at SOC 2018 (+ BLS group codes): `beta_gpt4`, `beta_human`, `n_tasks`, `how` (exact / pooled from parts / not rated).

In [10]:
print(f"stg_eloundou__task_exposure: {len(tasks):,} rows ({tasks['O*NET-SOC Code'].nunique()} O*NET jobs)")
print(f"stg_eloundou__job_exposure:  {len(occ):,} rows (published job scores, kept for the tie-out)")
print(f"SOC-level score in core: {soc_b.size} SOC jobs + BLS group codes via their parts")

stg_eloundou__task_exposure: 19,265 rows (923 O*NET jobs)
stg_eloundou__job_exposure:  923 rows (published job scores, kept for the tie-out)
SOC-level score in core: 798 SOC jobs + BLS group codes via their parts
